# AxonPath tutorial: from annotated images to measurements

This notebook walks through the **complete AxonPath workflow**, from data preparation to model export in BioImage.IO format.

You will learn how to:
1. Prepare datasets for training
2. Train an AxonPath model
3. Evaluate model performance on a test dataset
4. Run inference on new images
5. Measure fibre, inner cylinder and axon morphometry (g-ratios, shape, spatial metrics)
6. Export the trained model as a **BioImage.IO package** for the QuPath extension

Run the notebook from the `notebooks/` folder, in the `axonpath` conda environment (see the README's *Installation* section).

Sections 1–3 need your own annotated dataset, and a GPU is strongly recommended for training. Sections 4 and 5 run on the example images included in the repository with the pretrained weights, so you can start there.

## 1. Data preparation

Before training, the raw datasets are tiled and split into train, validation and test subsets.
This step also computes the skeleton-aware distance transforms (SDTs) used as training targets.

`split_dataset()` expects the dataset root folder to follow this structure:

```
datasets/
  dataset1/
    images/    # the images
    masks/     # semantic masks: 0 background, 1 myelin, 2 inner tongue, 3 axon
    labels/    # fibre instance labels: one integer ID per fibre
```

Each image in `images/` must have a mask and a label image with **exactly the same file name**.
For training, values 2 and 3 together form the inner cylinder. Brightfield masks have no inner tongue (values 0, 1 and 3 only).

If your data are annotated in QuPath, the AxonPath extension exports them in this format:
see [Export annotations → Training mode](https://github.com/paucabar/qupath-extension-axonpath/blob/main/docs/export.md#training-mode).

In [ ]:
from axonpath.data_preparation.data_preparation import split_dataset

In [ ]:
# Prepare dataset tiles and splits for training
split_dataset(
    in_root = "datasets_em",
    out_root = "prepared_data_em",
    create_test_split = True,
)

## 2. Model training

Training is configured with a `TrainingConfig` object: learning rate, batch size, number of epochs, the
training and validation folders, and optional pretrained weights to fine-tune from.

`min_diameter` is the minimum fibre diameter **in pixels**, used to discard small objects when the
validation predictions are scored. The released models use 20 for EM and 15 for brightfield.

Training writes to `output_dir`:
- the model checkpoint (`model_checkpoint.pth.tar`), used to resume training
- the best weights (`best_weights_model.pth`) and the last-epoch weights (`last_epoch_model.pth`)
- training and validation loss plots (`loss_plot.pdf`, `loss_plot_log.pdf`)
- a validation segmentation scores plot (`segmentation_scores_plot`)

In [ ]:
import torch
from axonpath.training.config import TrainingConfig
from axonpath.training.train import train

In [ ]:
# Define training configuration
train_config = TrainingConfig(
    learning_rate = 1e-3,
    batch_size = 8,
    num_epochs = 1000,
    num_workers = 0,
    image_height = 512,
    image_width = 512,
    pin_memory = True,
    device = "cuda" if torch.cuda.is_available() else "cpu",
    fibre_threshold = 0.5,
    axon_threshold = 0.5,
    min_diameter = 20.0,           # pixels; 20 for EM, 15 for brightfield
    train_dir = "prepared_data_em/train_tiles/",
    val_dir = "prepared_data_em/val_tiles/",
    pretrained_weights = None,     # optional: "em", "bf" or the path to a .pth file, to fine-tune from
    load_checkpoint = False,       # resume from the checkpoint in output_dir if True
    model_name = "axonpath",
    output_dir = ".",
    )

In [ ]:
# Train the model
%matplotlib inline
train(train_config)

## 3. Model evaluation

After training, evaluate the model on the test set. Predicted fibres, inner cylinders and axons are
matched to the ground truth at IoU thresholds from 0.5 to 0.9, and **F1**, **precision** and **recall**
are computed at each threshold.

Results are saved as `.tsv` files and can be plotted as summary bar charts.

In [ ]:
import torch
from axonpath.evaluation.evaluate_model import evaluate_model_on_testset, compute_imagewise_means, plot_summary_bar

In [ ]:
test_dir = "prepared_data_em/test_tiles/"
model_path = "best_weights_model.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"
min_diameter = 20.0  # pixels; use the same value as for training

%matplotlib inline
fibre_results, axon_results, inner_cylinder_results = evaluate_model_on_testset(
    model_path,
    test_dir,
    device,
    min_diameter,
    output_csv = "Evaluation_Results",
    display_figure = True,  # display ground truth and predicted labels
    show_plots = True,      # display F1 at IoU thresholds 0.5-0.9
)

In [ ]:
# Plot overall mean metrics for each structure
mean_metrics_fibre = compute_imagewise_means(fibre_results)
plot_summary_bar(mean_metrics_fibre, "Fibre")

mean_metrics_inner_cylinder = compute_imagewise_means(inner_cylinder_results)
plot_summary_bar(mean_metrics_inner_cylinder, "Inner cylinder")

mean_metrics_axon = compute_imagewise_means(axon_results)
plot_summary_bar(mean_metrics_axon, "Axon")

## 4. Inference on new images

Inference uses the pretrained weights (`em`, `bf`) or your own `.pth` file. It returns:
- fibre instance labels, from the fibre distance head
- axon instance labels, from the axon distance head
- inner cylinder instance labels, from the semantic head (EM only; `None` when `predict_inner_cylinder=False`)
- the semantic map (0 background, 1 myelin, 2 inner cylinder)

Axons and inner cylinders are given the ID of the fibre that contains them.

Choose one of the presets below. They match the three released models and their example images,
which are already at the model's pixel size. To use your own images, **rescale them to the model's
pixel size first**: the weights only work well at the resolution they were trained at.

In brightfield images the inner tongue cannot be resolved, so the brightfield preset sets
`predict_inner_cylinder=False`, as in the exported model: only fibres and axons are segmented, and
the axon includes the inner tongue.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from axonpath.inference import load_image, load_model, run_inference

In [ ]:
EXAMPLE_DIR = Path("..") / "axonpath" / "example_images"

# One preset per released model (see the README's "Pretrained models" section)
PRESETS = {
    "em_cns": dict(
        weights = "em",
        image = "exp69_72_EM10N_NOLPC_P5_0004_roi0_crop1022.tif",
        pixel_size_um = 0.008,
        min_diameter = 20.0,
        predict_inner_cylinder = True,
    ),
    "em_pns": dict(
        weights = "em",
        image = "L51L1_4_roi0.tif",
        pixel_size_um = 0.03,
        min_diameter = 20.0,
        predict_inner_cylinder = True,
    ),
    "brightfield": dict(
        weights = "bf",
        image = "A72E_roi0_crop1500.tif",
        pixel_size_um = 0.071,
        min_diameter = 15.0,
        predict_inner_cylinder = False,
    ),
}

preset = PRESETS["em_cns"]  # <-- "em_cns", "em_pns" or "brightfield"

# Load input image (normalised to the 1st-99th percentile) and model.
# For your own model, set model_path to its .pth file.
image = load_image(EXAMPLE_DIR / preset["image"])
model = load_model(preset["weights"])

# Run inference
labels_fibre, labels_axon, labels_inner_cylinder, semantic = run_inference(
    image = image,
    model = model,
    fibre_threshold = 0.5,
    axon_threshold = 0.5,
    min_diameter = preset["min_diameter"],
    predict_inner_cylinder = preset["predict_inner_cylinder"],
)

In [ ]:
# Visualize results
fig, axs = plt.subplots(1, 4, figsize=(18, 6))

axs[0].imshow(image, cmap='gray')
axs[0].set_title("Input Image")
axs[0].axis('off')

axs[1].imshow(semantic, cmap='viridis')
axs[1].set_title("Semantic Segmentation")
axs[1].axis('off')

axs[2].imshow(labels_fibre, cmap='nipy_spectral', interpolation="nearest")
axs[2].set_title("Fibre Instances")
axs[2].axis('off')

axs[3].imshow(labels_axon, cmap='nipy_spectral', interpolation="nearest")
axs[3].set_title("Axon Instances")
axs[3].axis('off')

plt.tight_layout()
plt.show()

## 5. Measure

`metrics_table()` computes one row of measurements per fibre:
- area of the fibre, inner cylinder and axon
- myelin g-ratio (inner cylinder diameter / fibre diameter; EM only) and axon g-ratio (axon diameter / fibre diameter), from equivalent circular diameters
- fibre circularity, solidity, eccentricity and major axis length
- distance from the fibre border to the nearest other fibre, and from its centroid to the nearest other centroid
- local confluence: the fraction of a 512 × 512 pixel window around the fibre that is covered by other fibres

As in the QuPath extension, fibres without an axon (and, in EM, without an inner cylinder) are
discarded, and brightfield results have no inner cylinder columns.

In [ ]:
import numpy as np

from axonpath.measure.axon_metrics import metrics_table
from axonpath.utils.label_ops import remove_unmapped_labels

In [ ]:
# Keep only fibres that contain an axon (and, in EM, an inner cylinder), as the QuPath extension does
fibres = remove_unmapped_labels(labels_fibre, labels_axon)
if labels_inner_cylinder is not None:
    fibres = remove_unmapped_labels(fibres, labels_inner_cylinder)
    inner_cylinder = labels_inner_cylinder
else:
    # Brightfield: no inner cylinders
    inner_cylinder = np.zeros_like(fibres)

df = metrics_table(fibres, inner_cylinder, pixel_size_um=preset["pixel_size_um"], axon_labels=labels_axon)

if labels_inner_cylinder is None:
    # Without inner cylinders these columns are not meaningful
    df = df.drop(columns=["InnerCylinder Area (µm²)", "Myelin g-ratio"])

df.head()

## 6. Export as a BioImage.IO model

Finally, export the trained model in [BioImage.IO](https://bioimage.io) format, ready for the
AxonPath QuPath extension: place the output folder in the extension's model directory.

The export needs:
- the model name and version
- the pixel size (µm) the model was trained at and its minimum fibre diameter (pixels)
- whether it predicts inner cylinders (`True` for EM, `False` for brightfield)
- a test image at the model's pixel size, used to generate the test input and output and the cover image
- authors, license and, optionally, a citation and DOI

In [ ]:
from axonpath.export_utils.create_bioimageio_model import export_bioimageio
from axonpath.export_utils.config import BioimageioExportConfig

In [ ]:
# Configure export metadata (replace with the details of your model)
bioimageio_config = BioimageioExportConfig(
    model_path = "best_weights_model.pth",
    model_name = "my_axonpath_model",
    model_version = "0.1.0",
    test_img_path = str(EXAMPLE_DIR / "exp69_72_EM10N_NOLPC_P5_0004_roi0_crop1022.tif"),
    output_dir = "my_axonpath_model-0.1.0",
    model_pixel_size = 0.008,
    min_diameter = 20.0,
    predict_inner_cylinder = True,
    description = "AxonPath model for segmenting myelinated fibres, inner cylinders and axons in my EM images.",
    author_names = ["Your Name"],
    citation_text = None,  # optional, e.g. a publication describing your model
    citation_doi = None,   # optional; without a DOI the citation links to the AxonPath repository
    license_id = "CC-BY-4.0",
    validate = True,       # check the package against the BioImage.IO specification
)

In [ ]:
# Run export
export_bioimageio(bioimageio_config)